In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt
import os
import warnings
warnings.filterwarnings('ignore')

thunderbolts_base_dir = 'C:/Users/Maks/Desktop/Jupyter/thunderbolts_data/hdf'
years = [2012, 2013, 2014, 2015, 2016, 2017, 2018]

output_dir = 'C:/Users/Maks/Desktop/Jupyter/statistics_vereya'
os.makedirs(output_dir, exist_ok=True)

print("=" * 70)
print("ЗАГРУЗКА ДАННЫХ ВЕРЕЯ-М за 2012-2018 гг.")
print("=" * 70)

all_data = []
for year in years:
    file_path = os.path.join(thunderbolts_base_dir, f'{year}_thunderbolts_clastered.h5')
    if os.path.exists(file_path):
        df_year = pd.read_hdf(file_path, 'strikes')
        df_year['year'] = year
        all_data.append(df_year)
        print(f"{year}: загружено {len(df_year):,} разрядов, {df_year['clnb'].nunique()} кластеров")
    else:
        print(f"{year}: файл не найден")
df = pd.concat(all_data, ignore_index=False)
print(f"\nВСЕГО: {len(df):,} разрядов, {df['clnb'].nunique()} кластеров")
if not pd.api.types.is_datetime64_any_dtype(df.index):
    df.index = pd.to_datetime(df.index)
df['hour_utc'] = df.index.hour

# ============= СОЗДАНИЕ CSV СО СТАТИСТИКОЙ ПО КЛАСТЕРАМ =============
print("\n" + "=" * 70)
print("СОЗДАНИЕ CSV ФАЙЛА СО СТАТИСТИКОЙ КЛАСТЕРОВ")
print("=" * 70)

clusters_positive = df[(df['clnb'] > 0)].groupby(['year', 'clnb'])
cluster_stats_list = []
for (year, clnb), group in clusters_positive:
    # Временные метки
    start_time = group.index.min()
    end_time = group.index.max()
    duration = end_time - start_time
    duration_seconds = duration.total_seconds()
    duration_minutes = duration_seconds / 60
    # Координаты
    lat_min = group['lat'].min()
    lat_max = group['lat'].max()
    lon_min = group['lon'].min()
    lon_max = group['lon'].max()
    # Площадь (кв. км)
    center_lat_for_area = (lat_min + lat_max) / 2
    width_deg = abs(lon_max - lon_min)
    height_deg = abs(lat_max - lat_min)
    width_km = width_deg * 111 * np.cos(np.radians(center_lat_for_area))
    height_km = height_deg * 111
    area_km2 = width_km * height_km
    # Центр кластера
    center_lat = group['lat'].mean()
    center_lon = group['lon'].mean()
    if center_lon < 0:
        center_lon_360 = center_lon + 360
    else:
        center_lon_360 = center_lon
    # Количество разрядов
    n_strikes = len(group)
    cluster_stats_list.append({
        'clnb': clnb,
        'year': year,
        'duration_minutes': duration_minutes,
        'area_km2': area_km2,
        'center_lat': center_lat,
        'center_lon': center_lon,
        'center_lon_360': center_lon_360,
        'n_strikes': n_strikes
    })
df_clusters = pd.DataFrame(cluster_stats_list)
df_clusters = df_clusters.sort_values(['year', 'clnb']).reset_index(drop=True)

# Сохранение в CSV
csv_path = os.path.join(output_dir, 'clusters_statistics.csv')
df_clusters.to_csv(csv_path, index=False, encoding='utf-8')
print(f"\nCSV файл сохранен: {csv_path}")
print(f"Всего кластеров (clnb > 0): {len(df_clusters):,}")

# Выводим базовую статистику для проверки
print("\n" + "=" * 70)
print("СТАТИСТИКА ПО КЛАСТЕРАМ")
print("=" * 70)
print(f"Длительность кластеров (минуты):")
print(f"  Медиана: {df_clusters['duration_minutes'].median():.1f} мин")
print(f"  Среднее: {df_clusters['duration_minutes'].mean():.1f} мин")
print(f"  Мин: {df_clusters['duration_minutes'].min():.1f} мин")
print(f"  Макс: {df_clusters['duration_minutes'].max():.1f} мин")
print(f"\nПлощадь кластеров (км²):")
print(f"  Медиана: {df_clusters['area_km2'].median():.1f} км²")
print(f"  Среднее: {df_clusters['area_km2'].mean():.1f} км²")
print(f"  Мин: {df_clusters['area_km2'].min():.1f} км²")
print(f"  Макс: {df_clusters['area_km2'].max():.1f} км²")

# ============= ДОПОЛНИТЕЛЬНЫЙ CSV: ПОЧАСОВАЯ СТАТИСТИКА РАЗРЯДОВ =============
print("\n" + "=" * 70)
print("СОЗДАНИЕ CSV ФАЙЛА С ПОЧАСОВОЙ СТАТИСТИКОЙ РАЗРЯДОВ")
print("=" * 70)

hourly_stats = []
for hour in range(24):
    df_hour = df[df['hour_utc'] == hour]
    df_hour_neg = df_hour[df_hour['amp'] < 0]
    df_hour_pos = df_hour[df_hour['amp'] > 0]
    hourly_stats.append({
        'hour_utc': hour,
        'total_strikes': len(df_hour),
        'negative_strikes': len(df_hour_neg),
        'positive_strikes': len(df_hour_pos)
    })
df_hourly = pd.DataFrame(hourly_stats)
hourly_csv_path = os.path.join(output_dir, 'hourly_statistics.csv')
df_hourly.to_csv(hourly_csv_path, index=False, encoding='utf-8')
print(f"CSV файл сохранен: {hourly_csv_path}")

# ============= ДОПОЛНИТЕЛЬНЫЙ CSV: ГОДОВАЯ СТАТИСТИКА =============
print("\n" + "=" * 70)
print("СОЗДАНИЕ CSV ФАЙЛА С ГОДОВОЙ СТАТИСТИКОЙ")
print("=" * 70)

yearly_stats = []
for year in years:
    df_year = df[df['year'] == year]
    
    yearly_stats.append({
        'year': year,
        'total_strikes': len(df_year)
    })
df_yearly = pd.DataFrame(yearly_stats)
yearly_csv_path = os.path.join(output_dir, 'yearly_statistics.csv')
df_yearly.to_csv(yearly_csv_path, index=False, encoding='utf-8')
print(f"CSV файл сохранен: {yearly_csv_path}")

# ============= ПОСТРОЕНИЕ ГРАФИКОВ ИЗ CSV ДАННЫХ =============
print("\n" + "=" * 70)
print("ПОСТРОЕНИЕ ГРАФИКОВ")
print("=" * 70)

# Загружаем CSV файлы для построения графиков
df_clusters = pd.read_csv(csv_path)
df_hourly = pd.read_csv(hourly_csv_path)
df_yearly = pd.read_csv(yearly_csv_path)

# ============= 1. РАСПРЕДЕЛЕНИЕ ПО ВРЕМЕНИ СУТОК =============
print("  1. Временное распределение...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1.1. Все разряды по часам UTC
ax1 = axes[0]
ax1.bar(df_hourly['hour_utc'], df_hourly['total_strikes'], color='steelblue', edgecolor='black', alpha=0.7)
ax1.set_xlabel('Время (UTC)', fontsize=12)
ax1.set_ylabel('Количество разрядов', fontsize=12)
ax1.set_title(f'Распределение всех разрядов по времени\n(всего: {df_hourly["total_strikes"].sum():,} разрядов)', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(range(0, 24, 2))

# 1.2. Разряды по знаку
ax2 = axes[1]
width = 0.35
x = np.arange(24)
ax2.bar(x - width/2, df_hourly['negative_strikes'], width, label='Отрицательные', color='blue', alpha=0.7)
ax2.bar(x + width/2, df_hourly['positive_strikes'], width, label='Положительные', color='red', alpha=0.7)
ax2.set_xlabel('Время (UTC)', fontsize=12)
ax2.set_ylabel('Количество разрядов', fontsize=12)
ax2.set_title('Распределение разрядов различного знака амплитуды по времени', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.savefig(os.path.join(output_dir, '1_time_distribution.png'), dpi=300, bbox_inches='tight')
plt.close()

# ============= 2. РАСПРЕДЕЛЕНИЕ ПО ДЛИТЕЛЬНОСТИ =============
print("  2. Распределение по длительности...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

durations_min = df_clusters['duration_minutes']

# 2.1. Линейная шкала
ax1 = axes[0]
ax1.hist(durations_min, bins=50, color='coral', edgecolor='black', alpha=0.7)
ax1.set_xlabel('Длительность кластера (минуты)', fontsize=12)
ax1.set_ylabel('Количество кластеров', fontsize=12)
ax1.set_title('Распределение кластеров по длительности', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.axvline(x=durations_min.median(), color='red', linestyle='--', linewidth=2, label=f'Медиана: {durations_min.median():.1f} мин')
ax1.legend()

# 2.2. Логарифмическая шкала по Y
ax2 = axes[1]
ax2.hist(durations_min, bins=50, color='coral', edgecolor='black', alpha=0.7, log=True)
ax2.set_xlabel('Длительность кластера (минуты)', fontsize=12)
ax2.set_ylabel('Количество кластеров (лог. шкала)', fontsize=12)
ax2.set_title('Распределение кластеров по длительности\n(логарифмическая шкала)', fontsize=12)
ax2.grid(True, alpha=0.3, axis='y')
ax2.axvline(x=durations_min.median(), color='red', linestyle='--', linewidth=2, label=f'Медиана: {durations_min.median():.1f} мин')
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(output_dir, '2_duration_distribution.png'), dpi=300, bbox_inches='tight')
plt.close()

# ============= 3. РАСПРЕДЕЛЕНИЕ ПО ПЛОЩАДИ =============
print("  3. Распределение по площади...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

area_km2 = df_clusters['area_km2']
area_filtered = area_km2[area_km2 < area_km2.quantile(0.99)]

# 3.1. Линейная шкала
ax1 = axes[0]
ax1.hist(area_filtered, bins=50, color='lightgreen', edgecolor='black', alpha=0.7)
ax1.set_xlabel('Площадь кластера (кв. км)', fontsize=12)
ax1.set_ylabel('Количество кластеров', fontsize=12)
ax1.set_title('Распределение кластеров по площади', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.axvline(x=area_km2.median(), color='red', linestyle='--', linewidth=2, label=f'Медиана: {area_km2.median():.0f} км²')
ax1.legend()

# 3.2. Логарифмическая шкала только по Y
ax2 = axes[1]
area_for_log = area_km2[area_km2 > 0]
ax2.hist(area_for_log, bins=50, color='lightgreen', edgecolor='black', alpha=0.7, log=True)
ax2.set_xlabel('Площадь кластера (кв. км)', fontsize=12)
ax2.set_ylabel('Количество кластеров (лог. шкала)', fontsize=12)
ax2.set_title('Распределение кластеров по площади\n(логарифмическая шкала)', fontsize=12)
ax2.grid(True, alpha=0.3, axis='y')
ax2.axvline(x=area_km2.median(), color='red', linestyle='--', linewidth=2, label=f'Медиана: {area_km2.median():.0f} км²')
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(output_dir, '3_area_distribution.png'), dpi=300, bbox_inches='tight')
plt.close()

# ============= 4. ШИРОТНОЕ И ДОЛГОТНОЕ РАСПРЕДЕЛЕНИЕ =============
print("  4. Широтное и долготное распределение...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

lat_bins = np.arange(-90, 91, 5)
lon_bins = np.arange(0, 361, 10)

# Нормализуем долготу для отображения (0-360)
lon_positive = df['lon'].copy()
lon_positive[lon_positive < 0] += 360

# ===== ВЕРХНИЙ РЯД: ЛИНЕЙНАЯ ШКАЛА =====

# 4.1. Широтное распределение (линейная шкала)
ax1 = axes[0, 0]
ax1.hist(df['lat'], bins=lat_bins, color='skyblue', edgecolor='black', alpha=0.7)
ax1.set_xlabel('Широта (градусы)', fontsize=12)
ax1.set_ylabel('Количество разрядов', fontsize=12)
ax1.set_title('Широтное распределение', fontsize=12)
ax1.grid(True, alpha=0.3)

# 4.2. Долготное распределение (линейная шкала)
ax2 = axes[0, 1]
ax2.hist(lon_positive, bins=lon_bins, color='lightcoral', edgecolor='black', alpha=0.7)
ax2.set_xlabel('Долгота (градусы)', fontsize=12)
ax2.set_ylabel('Количество разрядов', fontsize=12)
ax2.set_title('Долготное распределение', fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(range(0, 361, 60))

# ===== НИЖНИЙ РЯД: ЛОГАРИФМИЧЕСКАЯ ШКАЛА =====
# 4.3. Широтное распределение (логарифмическая шкала)
ax3 = axes[1, 0]
ax3.hist(df['lat'], bins=lat_bins, color='skyblue', edgecolor='black', alpha=0.7, log=True)
ax3.set_xlabel('Широта (градусы)', fontsize=12)
ax3.set_ylabel('Количество разрядов (лог. шкала)', fontsize=12)
ax3.set_title('Широтное распределение (логарифмическая шкала)', fontsize=12)
ax3.grid(True, alpha=0.3, axis='y')

# 4.4. Долготное распределение (логарифмическая шкала)
ax4 = axes[1, 1]
ax4.hist(lon_positive, bins=lon_bins, color='lightcoral', edgecolor='black', alpha=0.7, log=True)
ax4.set_xlabel('Долгота (градусы)', fontsize=12)
ax4.set_ylabel('Количество разрядов (лог. шкала)', fontsize=12)
ax4.set_title('Долготное распределение (логарифмическая шкала)', fontsize=12)
ax4.grid(True, alpha=0.3, axis='y')
ax4.set_xticks(range(0, 361, 60))

plt.tight_layout()
plt.savefig(os.path.join(output_dir, '4_latitude_longitude_distribution.png'), dpi=300, bbox_inches='tight')
plt.close()

# ============= 5. ГОДОВАЯ СТАТИСТИКА =============
print("  5. Годовая статистика...")

fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(df_yearly['year'], df_yearly['total_strikes'], color='steelblue', edgecolor='black', alpha=0.7)
ax.set_xlabel('Год', fontsize=12)
ax.set_ylabel('Количество разрядов', fontsize=12)
ax.set_title('Количество разрядов по годам', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.set_xticks(years)

# Значения над столбцами
for i, (year, strikes) in enumerate(zip(df_yearly['year'], df_yearly['total_strikes'])):
    ax.text(year, strikes + max(df_yearly['total_strikes']) * 0.01, 
            f'{strikes:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, '5_yearly_statistics.png'), dpi=300, bbox_inches='tight')
plt.close()

# ============= ФИНАЛЬНЫЙ ВЫВОД =============
print("\n" + "=" * 70)
print(f"ВСЕ ДАННЫЕ СОХРАНЕНЫ В ПАПКЕ: {output_dir}")
print("=" * 70)
print("\nСозданные файлы:")
print(f"  1. clusters_statistics.csv - статистика по каждому кластеру")
print(f"  2. hourly_statistics.csv - почасовая статистика разрядов")
print(f"  3. yearly_statistics.csv - годовая статистика")
print(f"  4. 1_time_distribution.png - временное распределение (2 графика)")
print(f"  5. 2_duration_distribution.png - распределение по длительности (линейная и лог шкала по Y)")
print(f"  6. 3_area_distribution.png - распределение по площади (линейная и лог шкала по Y)")
print(f"  7. 4_latitude_longitude_distribution.png - широтное и долготное распределение")
print(f"  8. 5_yearly_statistics.png - количество разрядов по годам")
print("=" * 70)

ЗАГРУЗКА ДАННЫХ ВЕРЕЯ-М за 2012-2018 гг.
2012: загружено 658,721 разрядов, 5315 кластеров
2013: загружено 993,151 разрядов, 10704 кластеров
2014: загружено 1,735,799 разрядов, 15653 кластеров
2015: загружено 870,515 разрядов, 11684 кластеров
2016: загружено 537,429 разрядов, 8637 кластеров
2017: загружено 112,486 разрядов, 2017 кластеров
2018: загружено 161,059 разрядов, 2645 кластеров

ВСЕГО: 5,069,160 разрядов, 15844 кластеров

СОЗДАНИЕ CSV ФАЙЛА СО СТАТИСТИКОЙ КЛАСТЕРОВ

CSV файл сохранен: C:/Users/Maks/Desktop/Jupyter/statistics_vereya\clusters_statistics.csv
Всего кластеров (clnb > 0): 56,648

СТАТИСТИКА ПО КЛАСТЕРАМ
Длительность кластеров (минуты):
  Медиана: 57.6 мин
  Среднее: 89.5 мин
  Мин: 0.0 мин
  Макс: 1468.4 мин

Площадь кластеров (км²):
  Медиана: 7033.7 км²
  Среднее: 30544.1 км²
  Мин: 0.0 км²
  Макс: 3068348.6 км²

СОЗДАНИЕ CSV ФАЙЛА С ПОЧАСОВОЙ СТАТИСТИКОЙ РАЗРЯДОВ
CSV файл сохранен: C:/Users/Maks/Desktop/Jupyter/statistics_vereya\hourly_statistics.csv

СОЗДАНИЕ CSV